# IMACS Stroke/NVU PET candidate screen — 01 replicated acute dataset

**Dataset:** GSE174574, mouse MCAO at 24 h, 3 sham + 3 MCAO samples.

**Purpose:** test the first frozen PET/BBB candidate panel in a dataset with actual biological replication before using one-sample temporal atlases.

### /TEAMRED contract
- The biological replicate is the **sample/animal**, not the cell.
- Cell-level data are used for broad cell-type assignment and expression summaries.
- Differential expression is performed on **sample × cell-type pseudobulk counts**.
- RNA does not establish protein abundance, luminal accessibility, tracer binding, or PET feasibility.
- Host targets (APJ/APLNR and AMOT) receive no preferential statistical treatment.
- A negative result is useful; it can kill a bad project early.


In [ ]:
!pip -q install "scanpy>=1.10,<1.12" "anndata>=0.10" "pydeseq2==0.4.12" "leidenalg>=0.10" "igraph>=0.11" "openpyxl>=3.1"

import os, sys, gzip, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmread
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=90, facecolor="white")

ROOT = Path(os.environ.get("IMACS_ROOT", "/content/imacs_stroke_pet" if Path("/content").exists() else "imacs_stroke_pet"))
RAW = ROOT/"raw"
RESULTS = ROOT/"results"
FIGS = ROOT/"figures"
for p in (RAW, RESULTS, FIGS): p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Python:", sys.version.split()[0], "scanpy:", sc.__version__)


## Frozen candidate panel

The repository catalog is read from the same branch. The first screen focuses on biological reproducibility and cell compartment; PET targetability is a later gate.

In [ ]:
CATALOG_URL = "https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv"
CANDIDATES = pd.read_csv(CATALOG_URL).drop_duplicates("gene")
display(CANDIDATES)
CANDIDATES.to_csv(RESULTS/"00_frozen_candidate_catalog.csv", index=False)


## Download raw 10x matrices

GEO provides processed 10x count matrices for each of the six biological samples. We download the exact files and record SHA256 checksums.

In [ ]:
SAMPLES = pd.DataFrame([
    ["GSM5319987","sham1","Sham"],
    ["GSM5319988","sham2","Sham"],
    ["GSM5319989","sham3","Sham"],
    ["GSM5319990","MCAO1","MCAO"],
    ["GSM5319991","MCAO2","MCAO"],
    ["GSM5319992","MCAO3","MCAO"],
], columns=["gsm","prefix","condition"])
display(SAMPLES)

def geo_block(gsm):
    return gsm[:7] + "nnn"

def geo_url(gsm, prefix, filename):
    return f"https://ftp.ncbi.nlm.nih.gov/geo/samples/{geo_block(gsm)}/{gsm}/suppl/{gsm}_{prefix}_{filename}"

def download(url, path):
    path=Path(path)
    if path.exists() and path.stat().st_size>0: return path
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(path,"wb") as f:
            for chunk in r.iter_content(1<<20):
                if chunk: f.write(chunk)
    return path

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20), b""): h.update(chunk)
    return h.hexdigest()

def fetch_sample(gsm,prefix,sample_id,condition):
    d=RAW/gsm; d.mkdir(exist_ok=True)
    paths={}
    for fn in ["matrix.mtx.gz","barcodes.tsv.gz","genes.tsv.gz"]:
        u=geo_url(gsm,prefix,fn)
        p=download(u,d/f"{gsm}_{prefix}_{fn}")
        paths[fn]=p
    with gzip.open(paths["matrix.mtx.gz"],"rb") as f:
        X=mmread(f).tocsr().T
    with gzip.open(paths["barcodes.tsv.gz"],"rt") as f:
        barcodes=[x.rstrip() for x in f]
    feats=pd.read_csv(paths["genes.tsv.gz"],sep="\t",header=None,compression="gzip")
    symbols=feats.iloc[:,1].astype(str).values if feats.shape[1]>1 else feats.iloc[:,0].astype(str).values
    a=ad.AnnData(X=X)
    a.obs_names=[f"{sample_id}:{b}" for b in barcodes]
    a.var_names=pd.Index(symbols)
    a.var_names_make_unique()
    a.obs["sample_id"]=sample_id
    a.obs["condition"]=condition
    return a, paths


In [ ]:
objs=[]; prov=[]
for r in SAMPLES.itertuples(index=False):
    print("Loading",r.gsm,r.prefix)
    a,paths=fetch_sample(r.gsm,r.prefix,r.prefix,r.condition)
    objs.append(a)
    for kind,p in paths.items():
        prov.append({"gsm":r.gsm,"sample_id":r.prefix,"condition":r.condition,"file":p.name,"bytes":p.stat().st_size,"sha256":sha256(p)})

pd.DataFrame(prov).to_csv(RESULTS/"01_input_checksums.csv",index=False)
adata=ad.concat(objs,join="outer",merge="same",index_unique=None)
assert adata.obs_names.is_unique, "Duplicate cell identifiers across samples!"
print(adata)


## QC

The first pass uses conservative filters and reports pre/post counts. Upper outliers are removed within sample rather than imposing a single arbitrary maximum across all six libraries.

In [ ]:
adata.var["mt"]=adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata,qc_vars=["mt"],inplace=True)

before=adata.obs.groupby("sample_id").agg(
    n_cells=("sample_id","size"),
    median_genes=("n_genes_by_counts","median"),
    median_umis=("total_counts","median"),
    median_pct_mt=("pct_counts_mt","median"))
display(before)

keep=(adata.obs["n_genes_by_counts"].values>=200) & (adata.obs["pct_counts_mt"].values<20)
for s,names in adata.obs.groupby("sample_id").groups.items():
    idx=adata.obs_names.get_indexer(names)
    d=adata.obs.iloc[idx]
    keep[idx] &= d["n_genes_by_counts"].values <= d["n_genes_by_counts"].quantile(.995)
    keep[idx] &= d["total_counts"].values <= d["total_counts"].quantile(.995)

adata=adata[keep].copy()
adata.layers["counts"]=adata.X.copy()
after=adata.obs.groupby("sample_id").size().rename("n_cells_after")
audit=before.join(after)
display(audit)
audit.to_csv(RESULTS/"02_QC_audit.csv")


## Broad cell-type assignment

We deliberately classify only broad compartments needed for target screening. Candidate genes themselves are **not** used as lineage markers.

In [ ]:
MARKERS={
"Endothelial":["Cldn5","Pecam1","Kdr","Flt1","Esam","Emcn","Ramp2","Klf2"],
"Pericyte":["Pdgfrb","Rgs5","Kcnj8","Abcc9","Anpep","Cd248","Cspg4"],
"VSMC":["Acta2","Tagln","Myh11","Cnn1","Rgs5"],
"Astrocyte":["Aqp4","Slc1a3","Aldoc","Gja1","S100b","Aldh1l1"],
"Microglia":["P2ry12","Tmem119","C1qa","C1qb","Hexb","Cx3cr1"],
"Myeloid":["Ptprc","Lyz2","Tyrobp","Aif1","Fcgr3"],
"Neuron":["Rbfox3","Snap25","Syt1","Map2","Tubb3"],
"Oligodendrocyte":["Mbp","Plp1","Mog","Mag","Cnp"],
"OPC":["Pdgfra","Olig1","Olig2","Sox10"],
"VLMC_Fibroblast":["Col1a1","Col1a2","Dcn","Lum","Col3a1"],
"Ependymal_CP":["Ttr","Foxj1","Clic6"]
}

sc.pp.normalize_total(adata,target_sum=1e4)
sc.pp.log1p(adata)
adata.raw=adata.copy()

sc.pp.highly_variable_genes(adata,n_top_genes=2500,flavor="seurat",batch_key="sample_id")
sc.tl.pca(adata,n_comps=40,use_highly_variable=True)
sc.pp.neighbors(adata,n_neighbors=15,n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata,resolution=.6,key_added="leiden")

for lineage,genes in MARKERS.items():
    genes=[g for g in genes if g in adata.raw.var_names]
    sc.tl.score_genes(adata,genes,score_name=f"score_{lineage}",use_raw=True)

score_cols=[f"score_{x}" for x in MARKERS]
cs=adata.obs.groupby("leiden")[score_cols].mean()
cs["best"]=cs.idxmax(axis=1).str.replace("score_","",regex=False)
vals=np.sort(cs[score_cols].values,axis=1)
cs["margin"]=vals[:,-1]-vals[:,-2]
cs["broad_celltype"]=np.where(cs["margin"]>=.05,cs["best"],"Ambiguous")
adata.obs["broad_celltype"]=adata.obs["leiden"].map(cs["broad_celltype"]).astype("category")

display(cs.sort_index())
cs.to_csv(RESULTS/"03_cluster_annotation_scores.csv")
sc.pl.umap(adata,color=["sample_id","condition","broad_celltype"],wspace=.35,show=True)


In [ ]:
sanity=["Cldn5","Pecam1","Pdgfrb","Rgs5","Aqp4","Slc1a3","P2ry12","Tmem119","Mbp","Plp1","Snap25","Ptprc"]
sanity=[g for g in sanity if g in adata.raw.var_names]
sc.pl.dotplot(adata,sanity,groupby="broad_celltype",use_raw=True,standard_scale="var",dendrogram=False)

composition=pd.crosstab(adata.obs["sample_id"],adata.obs["broad_celltype"])
display(composition)
composition.to_csv(RESULTS/"04_celltype_counts_by_sample.csv")


## Candidate expression by compartment

These metrics are descriptive: mean log-normalized RNA and the fraction of cells with detected transcript.

In [ ]:
def expr_summary(a,genes):
    rows=[]; var={g:i for i,g in enumerate(a.var_names)}
    for g in genes:
        if g not in var:
            rows.append({"gene":g,"broad_celltype":"__MISSING__","mean_log1p":np.nan,"pct_detected":np.nan,"n_cells":0})
            continue
        j=var[g]
        for ct,names in a.obs.groupby("broad_celltype",observed=True).groups.items():
            idx=a.obs_names.get_indexer(names)
            v=a.X[idx,j]; v=np.asarray(v.toarray()).ravel() if sp.issparse(v) else np.asarray(v).ravel()
            rv=a.layers["counts"][idx,j]; rv=np.asarray(rv.toarray()).ravel() if sp.issparse(rv) else np.asarray(rv).ravel()
            rows.append({"gene":g,"broad_celltype":ct,"mean_log1p":float(v.mean()),
                         "pct_detected":float((rv>0).mean()*100),"n_cells":len(idx)})
    return pd.DataFrame(rows)

expr=expr_summary(adata,CANDIDATES["gene"].tolist())
expr.to_csv(RESULTS/"05_candidate_expression_by_celltype.csv",index=False)
display(expr[expr["broad_celltype"].isin(["Endothelial","Pericyte","Astrocyte"])].head(100))


## Animal-level pseudobulk DE

Counts are summed per **sample × cell type** and analyzed with DESeq2 logic through PyDESeq2. A compartment is skipped if any sample has too few recovered cells/counts.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

def pseudobulk(a,ct):
    sub=a[a.obs["broad_celltype"]==ct]
    rows=[]; labels=[]
    for s,names in sub.obs.groupby("sample_id").groups.items():
        idx=sub.obs_names.get_indexer(names)
        rows.append(np.asarray(sub.layers["counts"][idx].sum(axis=0)).ravel())
        labels.append(s)
    counts=pd.DataFrame(rows,index=labels,columns=sub.var_names).round().astype(int)
    meta=sub.obs.drop_duplicates("sample_id").set_index("sample_id")[["condition"]].loc[labels]
    cells=sub.obs.groupby("sample_id").size().reindex(labels)
    audit=pd.DataFrame({"n_cells":cells,"library_size":counts.sum(1)}).join(meta)
    return counts,meta,audit

def run_deseq(a,ct,min_cells=20,min_lib=10000):
    counts,meta,audit=pseudobulk(a,ct)
    if len(meta)!=6 or set(meta["condition"])!={"Sham","MCAO"}:
        raise ValueError("incomplete 3+3 design")
    if (audit["n_cells"]<min_cells).any(): raise ValueError("sample below min cell count")
    if (audit["library_size"]<min_lib).any(): raise ValueError("sample below min pseudobulk library size")
    keep=(counts>=10).sum(0)>=3
    dds=DeseqDataSet(counts=counts.loc[:,keep],metadata=meta,design_factors="condition",refit_cooks=True,n_cpus=2)
    dds.deseq2()
    st=DeseqStats(dds,contrast=["condition","MCAO","Sham"],n_cpus=2)
    st.summary()
    out=st.results_df.reset_index().rename(columns={"index":"gene"})
    out["celltype"]=ct
    return out,audit

import traceback
allres=[]; audits=[]; failures=[]
for ct in ["Endothelial","Astrocyte","Pericyte","VSMC"]:
    try:
        counts,meta,audit=pseudobulk(adata,ct)
        present=[g for g in CANDIDATES.gene if g in counts.columns]
        counts.loc[:,present].to_csv(RESULTS/f"06_{ct}_raw_candidate_pseudobulk_counts.csv")
        r,a=run_deseq(adata,ct)
        allres.append(r); a["celltype"]=ct; audits.append(a.reset_index())
        r.to_csv(RESULTS/f"06_DESeq2_{ct}.csv",index=False)
    except Exception as e:
        failures.append({"celltype":ct,"error_type":type(e).__name__,"message":str(e)})
        print("SKIP",ct,":",type(e).__name__,str(e))
        traceback.print_exc(limit=3)

pd.DataFrame(failures,columns=["celltype","error_type","message"]).to_csv(RESULTS/"06_DESeq2_failure_audit.csv",index=False)
de=pd.concat(allres,ignore_index=True) if allres else pd.DataFrame(columns=["gene","celltype","baseMean","log2FoldChange","lfcSE","pvalue","padj"])
if audits: pd.concat(audits).to_csv(RESULTS/"06_pseudobulk_audit.csv",index=False)
display(de.head() if len(de) else de)


## Frozen-candidate evidence matrix

This is the first result that matters. It does **not** choose a PET tracer. It asks whether the frozen candidate biology is detectable and reproducible enough at 24 h to deserve deeper temporal/spatial analysis.

In [ ]:
cand=expr.rename(columns={"broad_celltype":"celltype"}).merge(CANDIDATES,on="gene",how="left",validate="many_to_one")
if len(de):
    cand=cand.merge(de[["gene","celltype","baseMean","log2FoldChange","lfcSE","pvalue","padj"]],
                    on=["gene","celltype"],how="left",validate="many_to_one")
else:
    for col in ["baseMean","log2FoldChange","lfcSE","pvalue","padj"]:
        cand[col]=np.nan

cand["direction"]=np.select([cand["log2FoldChange"]>0,cand["log2FoldChange"]<0],["up","down"],default="untested")
cand["sample_level_inference"]=np.where(cand["padj"].notna(),"pseudobulk 3+3","not available")
cand.to_csv(RESULTS/"07_frozen_candidate_replicated_evidence.csv",index=False)
display(cand[cand["celltype"].isin(["Endothelial","Astrocyte","Pericyte","VSMC"])].sort_values(["celltype","padj"],na_position="last").head(100))
assert len(cand)>0
print("DESeq2 result rows:",len(de), "failures:", len(failures))


## Interpretation gate

Do **not** promote a candidate solely because it has a low adjusted p-value.

For the next notebook a candidate must be evaluated separately on:
1. replicated RNA direction in GSE174574,
2. day 1/3/7 trajectory in GSE227651,
3. core/penumbra localization in GSE233815,
4. rat/model consistency,
5. protein localization and blood-facing accessibility,
6. normal-organ burden,
7. ligand/binder/tracer chemistry,
8. perfusion/BBB-leakage confounding,
9. human relevance.

The next notebook will add the temporal dataset without pseudo-replicated statistics.


In [ ]:
import json
manifest={
  "primary_dataset":"GSE174574",
  "design":"3 sham vs 3 MCAO at 24h",
  "inference_unit":"sample/animal",
  "candidate_count":int(len(CANDIDATES)),
  "output_dir":str(RESULTS),
}
(RESULTS/"RUN_MANIFEST.json").write_text(json.dumps(manifest,indent=2))
print("DONE. Results:", RESULTS)
print(sorted(p.name for p in RESULTS.iterdir()))
